# 08 — Postprocesamiento Morfológico (DeepLabv3+)

Aplica el pipeline de postprocesamiento focalizado sobre las predicciones del modelo DeepLabv3+ en el test set completo.

**Clases tratadas (6 de 8):**

| Clase | Ops | Área mín |
|-------|-----|----------|
| Construccion | cierre 3×3 | 500 px |
| Bosque | cierre 5×5 | 1000 px |
| DepositoAgua | apertura 2×2 + cierre 5×5 | 200 px |
| ZonaDura | apertura 3×3 + cierre 3×3 | 300 px |
| Via | closing direccional [1×7]+[7×1] | 100 px |
| CabeceraMunicipal | cierre 3×3 | 10000 px |

**ZonaVerde y Cerca:** sin modificar (limitación estructural de sensor/datos).

**Artefactos generados en** `results/test_evaluation/deeplabv3_postprocessed/`

## 1. Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import json
from datetime import datetime

from scipy.ndimage import median_filter, binary_closing, binary_opening, label as ndi_label
from skimage.morphology import remove_small_objects, disk

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

from scripts.utils import (
    get_device, set_seed,
    create_dataloaders, get_validation_augmentation,
    SegmentationMetrics, N_CLASSES, CLASS_NAMES,
    PATCHES_DIR, PATCHES_INDEX, CHECKPOINTS_DIR, RESULTS_DIR
)

set_seed(42)
device = get_device()

OUTPUT_DIR = RESULTS_DIR / 'test_evaluation' / 'deeplabv3_postprocessed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Output: {OUTPUT_DIR}")

## 2. Pipeline de postprocesamiento

Función que recibe una máscara predicha `(H, W)` con enteros de clase y devuelve la máscara postprocesada.

In [ ]:
# CLASS_NAMES es un dict {int: str}, usamos el mapeo inverso
CLASS_NAMES_TO_ID = {v: k for k, v in CLASS_NAMES.items()}

CLS_CONSTRUCCION      = CLASS_NAMES_TO_ID['Construccion']
CLS_BOSQUE            = CLASS_NAMES_TO_ID['Bosque']
CLS_DEPOSITOAGUA      = CLASS_NAMES_TO_ID['DepositoAgua']
CLS_ZONAVERDE         = CLASS_NAMES_TO_ID['ZonaVerde']
CLS_ZONADURA          = CLASS_NAMES_TO_ID['ZonaDura']
CLS_VIA               = CLASS_NAMES_TO_ID['Via']
CLS_CERCA             = CLASS_NAMES_TO_ID['Cerca']
CLS_CABECERAMUNICIPAL = CLASS_NAMES_TO_ID['CabeceraMunicipal']

print("Indices de clase:")
for idx, name in CLASS_NAMES.items():
    print(f"  {idx:>2}  {name}")


In [ ]:
def _apply_class_ops(mask, cls_idx, struct_close=None, struct_open=None, area_min=None):
    """
    Aplica operaciones morfologicas a una clase en la mascara.
    Modifica la mascara in-place y retorna la misma.
    """
    binary = (mask == cls_idx)
    processed = binary.copy()

    if struct_open is not None:
        processed = binary_opening(processed, structure=struct_open)

    if struct_close is not None:
        processed = binary_closing(processed, structure=struct_close)

    if area_min is not None and area_min > 0:
        processed = remove_small_objects(processed.astype(bool), min_size=area_min)

    added   = processed & ~binary
    mask[added] = cls_idx

    removed = binary & ~processed
    mask[removed] = 0

    return mask


def postprocess_mask(mask_np):
    """
    Pipeline de postprocesamiento v2 - parametros ajustados para patches 512x512.

    Cambios respecto a v1:
    - CabeceraMunicipal: SIN area_min (fragmentos en bordes de parche son validos)
    - Bosque: cierre reducido a 3x3 (5x5 causaba confusion con otras clases verdes)
    - area_min reducidos globalmente (operamos en parches, no imagen completa)
    """
    m = mask_np.copy()

    # Paso 1: Filtro de ruido global (mediana 3x3)
    m = median_filter(m, size=3).astype(np.int64)

    # Paso 2: Construccion - cierre 3x3 + area>=100
    _apply_class_ops(m, CLS_CONSTRUCCION,
                     struct_close=np.ones((3, 3), dtype=bool),
                     area_min=100)

    # Paso 3: Bosque - cierre 3x3 + area>=200  (reducido de 5x5 / 1000)
    _apply_class_ops(m, CLS_BOSQUE,
                     struct_close=np.ones((3, 3), dtype=bool),
                     area_min=200)

    # Paso 4: DepositoAgua - apertura 2x2 + cierre 3x3 + area>=100
    _apply_class_ops(m, CLS_DEPOSITOAGUA,
                     struct_close=np.ones((3, 3), dtype=bool),
                     struct_open=np.ones((2, 2), dtype=bool),
                     area_min=100)

    # Paso 5: ZonaDura - apertura 2x2 + cierre 3x3 + area>=100
    _apply_class_ops(m, CLS_ZONADURA,
                     struct_close=np.ones((3, 3), dtype=bool),
                     struct_open=np.ones((2, 2), dtype=bool),
                     area_min=100)

    # Paso 6: Via - closing direccional [1x5]+[5x1] + area>=50
    binary_via = (m == CLS_VIA)
    closed_h   = binary_closing(binary_via, structure=np.ones((1, 5), dtype=bool))
    closed_v   = binary_closing(binary_via, structure=np.ones((5, 1), dtype=bool))
    processed_via = remove_small_objects((closed_h | closed_v).astype(bool), min_size=50)
    m[processed_via & ~binary_via] = CLS_VIA
    m[binary_via & ~processed_via] = 0

    # Paso 7: CabeceraMunicipal - cierre 3x3, SIN area_min
    # (es la clase dominante, sus fragmentos en bordes de parche son validos)
    _apply_class_ops(m, CLS_CABECERAMUNICIPAL,
                     struct_close=np.ones((3, 3), dtype=bool),
                     area_min=None)

    # ZonaVerde y Cerca: sin modificar
    return m


print("Pipeline v2 definido. Cambios:")
print("  Construccion:      cierre 3x3, area>=100 (antes 500)")
print("  Bosque:            cierre 3x3, area>=200 (antes 5x5 / 1000)")
print("  DepositoAgua:      apertura 2x2 + cierre 3x3, area>=100 (antes 5x5 / 200)")
print("  ZonaDura:          apertura 2x2 + cierre 3x3, area>=100 (antes 3x3 / 300)")
print("  Via:               closing dir [1x5]+[5x1], area>=50 (antes 1x7 / 100)")
print("  CabeceraMunicipal: cierre 3x3, SIN area_min (antes area>=10000)")


## 3. Cargar modelo DeepLabv3+ y DataLoader

In [ ]:
MODEL_NAME     = 'deeplabv3_resnet50_baseline'
CHECKPOINT_PATH = CHECKPOINTS_DIR / MODEL_NAME / 'best_checkpoint.pth'

model = smp.DeepLabV3Plus(
    encoder_name='resnet50',
    encoder_weights=None,
    in_channels=3,
    classes=N_CLASSES
)
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

print(f"Modelo: {MODEL_NAME}")
print(f"Checkpoint: epoch {ckpt['epoch']}, val mIoU = {ckpt['best_val_miou']:.4f}")

val_transform = get_validation_augmentation()
dataloaders = create_dataloaders(
    patches_dir=PATCHES_DIR,
    indices_csv=PATCHES_INDEX,
    batch_size=8,
    num_workers=0,
    train_transform=val_transform,
    val_transform=val_transform,
    use_balanced_sampler=False
)
test_loader = dataloaders['test']
print(f"Test set: {len(test_loader.dataset):,} patches")

## 4. Validación visual en 16 patches

Antes de correr todo el test, verificar que el pipeline produce resultados razonables visualmente.

In [ ]:
N_VISUAL = 16
vis_imgs, vis_gt, vis_before, vis_after = [], [], [], []

model.eval()
with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        preds  = torch.argmax(model(images), dim=1).cpu().numpy()
        for i in range(images.shape[0]):
            if len(vis_imgs) >= N_VISUAL:
                break
            vis_imgs.append(images[i].cpu().permute(1,2,0).numpy())
            vis_gt.append(masks[i].numpy())
            vis_before.append(preds[i])
            vis_after.append(postprocess_mask(preds[i]))
        if len(vis_imgs) >= N_VISUAL:
            break

print(f"{N_VISUAL} patches listos para visualización.")

In [ ]:
# Mostrar 8 patches: imagen | GT | antes | después
N_SHOW = 8
fig, axes = plt.subplots(N_SHOW, 4, figsize=(18, 4.5 * N_SHOW))
titles = ['Imagen', 'Ground Truth', 'DeepLabv3+\n(sin PP)', 'DeepLabv3+\n(con PP)']

for col, t in enumerate(titles):
    axes[0, col].set_title(t, fontsize=11, fontweight='bold')

for row in range(N_SHOW):
    img = vis_imgs[row]
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    axes[row, 0].imshow(img)
    axes[row, 1].imshow(vis_gt[row],     cmap='tab10', vmin=0, vmax=8)
    axes[row, 2].imshow(vis_before[row], cmap='tab10', vmin=0, vmax=8)
    axes[row, 3].imshow(vis_after[row],  cmap='tab10', vmin=0, vmax=8)

    for col in range(4):
        axes[row, col].axis('off')

plt.suptitle('Validación visual — Postprocesamiento (16 patches de muestra)', fontsize=13, fontweight='bold', y=1.001)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'visual_validation_subset.png', dpi=150, bbox_inches='tight')
plt.show()
print("Revisa las columnas 'sin PP' vs 'con PP'. Si se ven razonables, continúa con §5.")
print("Si hay artefactos, ajusta los kernels en la celda de postprocess_mask().")

## 5. Evaluación completa — test set (antes y después)

Loop único sobre los 18,991 patches: calcula métricas ANTES y DESPUÉS del postprocesamiento.

In [ ]:
metrics_before = SegmentationMetrics(num_classes=N_CLASSES, ignore_index=0, class_names=CLASS_NAMES)
metrics_after  = SegmentationMetrics(num_classes=N_CLASSES, ignore_index=0, class_names=CLASS_NAMES)

model.eval()
with torch.no_grad():
    for images, masks in tqdm(test_loader, desc="Inferencia + Postprocesamiento"):
        images_gpu = images.to(device)
        masks_gpu  = masks.to(device)

        # Predicción raw
        preds_raw = torch.argmax(model(images_gpu), dim=1)

        # Métricas ANTES
        metrics_before.update(preds_raw, masks_gpu)

        # Postprocesamiento (CPU, numpy)
        preds_np  = preds_raw.cpu().numpy()  # (B, H, W)
        preds_pp  = np.stack([postprocess_mask(preds_np[i]) for i in range(preds_np.shape[0])])

        # Métricas DESPUÉS
        preds_pp_gpu = torch.from_numpy(preds_pp).long().to(device)
        metrics_after.update(preds_pp_gpu, masks_gpu)

print("\nCompletado!")

## 6. Resultados — comparativa antes vs después

In [ ]:
sum_before = metrics_before.get_summary(detailed=True)
sum_after  = metrics_after.get_summary(detailed=True)

iou_before = metrics_before.get_iou_per_class()
iou_after  = metrics_after.get_iou_per_class()
f1_before  = metrics_before.get_f1_per_class()
f1_after   = metrics_after.get_f1_per_class()
prec_before = metrics_before.get_precision_per_class()
prec_after  = metrics_after.get_precision_per_class()
rec_before  = metrics_before.get_class_accuracy()
rec_after   = metrics_after.get_class_accuracy()

CLASSES = [CLASS_NAMES[i] for i in range(1, N_CLASSES)]

print("=" * 80)
print("COMPARATIVA: ANTES vs DESPUÉS DEL POSTPROCESAMIENTO")
print("=" * 80)
print(f"{'Clase':<22} {'IoU Antes':>10} {'IoU Después':>12} {'Δ IoU':>8} {'Mejor':>6}")
print("-" * 64)
for i in range(1, N_CLASSES):
    before = iou_before[i]
    after  = iou_after[i]
    delta  = after - before
    tag    = '✓' if delta > 0.001 else ('=' if abs(delta) <= 0.001 else '✗')
    print(f"{CLASS_NAMES[i]:<22} {before:>10.4f} {after:>12.4f} {delta:>+8.4f} {tag:>6}")
print("-" * 64)
print(f"{'mIoU':<22} {sum_before['mIoU']:>10.4f} {sum_after['mIoU']:>12.4f} "
      f"{sum_after['mIoU'] - sum_before['mIoU']:>+8.4f}")
print(f"{'Pixel Acc':<22} {sum_before['pixel_accuracy']:>10.4f} {sum_after['pixel_accuracy']:>12.4f} "
      f"{sum_after['pixel_accuracy'] - sum_before['pixel_accuracy']:>+8.4f}")
print("=" * 80)

## 7. Guardar métricas postprocesadas

In [ ]:
dice_after = metrics_after.get_dice_per_class()

results_pp = {
    'model_name': MODEL_NAME,
    'postprocessing': True,
    'postprocessing_classes': ['Construccion', 'Bosque', 'DepositoAgua', 'ZonaDura', 'Via', 'CabeceraMunicipal'],
    'excluded_classes': ['ZonaVerde', 'Cerca'],
    'test_date': datetime.now().isoformat(),
    'test_patches': len(test_loader.dataset),
    'metrics_before': {
        'mIoU': float(sum_before['mIoU']),
        'pixel_accuracy': float(sum_before['pixel_accuracy'])
    },
    'metrics_after': {
        'mIoU': float(sum_after['mIoU']),
        'pixel_accuracy': float(sum_after['pixel_accuracy'])
    },
    'delta_miou': float(sum_after['mIoU'] - sum_before['mIoU']),
    'per_class': {}
}

for i in range(N_CLASSES):
    results_pp['per_class'][CLASS_NAMES[i]] = {
        'IoU_before':  float(iou_before[i]),
        'IoU_after':   float(iou_after[i]),
        'delta_IoU':   float(iou_after[i] - iou_before[i]),
        'F1_before':   float(f1_before[i]),
        'F1_after':    float(f1_after[i]),
        'Prec_before': float(prec_before[i]),
        'Prec_after':  float(prec_after[i]),
        'Rec_before':  float(rec_before[i]),
        'Rec_after':   float(rec_after[i]),
    }

# También guardar el JSON con el mismo formato que los otros modelos (para reutilizar en comparativas)
results_pp['metrics'] = {'mIoU': float(sum_after['mIoU']), 'pixel_accuracy': float(sum_after['pixel_accuracy'])}
results_pp['per_class_iou'] = {CLASS_NAMES[i]: {'IoU': float(iou_after[i])} for i in range(N_CLASSES)}

json_path = OUTPUT_DIR / 'test_metrics.json'
with open(json_path, 'w') as f:
    json.dump(results_pp, f, indent=2)

np.save(OUTPUT_DIR / 'confusion_matrix_before.npy', metrics_before.confusion_matrix)
np.save(OUTPUT_DIR / 'confusion_matrix_after.npy',  metrics_after.confusion_matrix)

print(f"Guardado: {json_path}")
print(f"mIoU antes:  {sum_before['mIoU']:.4f}")
print(f"mIoU después: {sum_after['mIoU']:.4f}  (+{sum_after['mIoU'] - sum_before['mIoU']:.4f})")

## 8. Figura 1 — IoU antes vs después por clase

In [ ]:
ious_b = [iou_before[i] for i in range(1, N_CLASSES)]
ious_a = [iou_after[i]  for i in range(1, N_CLASSES)]

x     = np.arange(len(CLASSES))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))

bars_b = ax.bar(x - width/2, ious_b, width, label=f'Sin PP (mIoU={sum_before["mIoU"]:.4f})',
                color='#95a5a6', edgecolor='black', linewidth=0.7)
bars_a = ax.bar(x + width/2, ious_a, width, label=f'Con PP (mIoU={sum_after["mIoU"]:.4f})',
                color='#2ecc71', edgecolor='black', linewidth=0.7)

ax.axhline(y=0.5, color='#27ae60', linestyle='--', alpha=0.7, linewidth=1.2, label='IoU = 0.5')
ax.axhline(y=0.45, color='#f39c12', linestyle=':', alpha=0.7, linewidth=1.0, label='Meta ≥ 0.45')

# Flechas de mejora
for i, (b, a) in enumerate(zip(ious_b, ious_a)):
    delta = a - b
    if delta > 0.002:
        ax.annotate('', xy=(x[i] + width/2, a + 0.01), xytext=(x[i] - width/2, b + 0.01),
                    arrowprops=dict(arrowstyle='->', color='#27ae60', lw=1.2))
    ax.text(x[i] + width/2, a + 0.02, f'{a:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=35, ha='right', fontsize=11)
ax.set_ylabel('IoU', fontsize=12)
ax.set_title('IoU por Clase — Antes vs Después del Postprocesamiento\n(DeepLabv3+ ResNet-50, Test Set)',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_iou_antes_despues.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR / 'fig_iou_antes_despues.png'}")

## 9. Figura 2 — Matriz de confusión antes vs después

In [ ]:
def norm_cm(cm):
    cm = cm.astype('float')
    row_sums = cm.sum(axis=1, keepdims=True)
    return np.where(row_sums > 0, cm / row_sums, 0.0)

cm_b = norm_cm(metrics_before.confusion_matrix)[1:, 1:]
cm_a = norm_cm(metrics_after.confusion_matrix)[1:, 1:]
labels = [c.replace('CabeceraMunicipal', 'Cabecera') for c in CLASSES]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, cm, title in zip(axes,
    [cm_b, cm_a],
    [f'Sin Postprocesamiento\nmIoU = {sum_before["mIoU"]:.4f}',
     f'Con Postprocesamiento\nmIoU = {sum_after["mIoU"]:.4f}']):
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                ax=ax, vmin=0, vmax=1, linewidths=0.3,
                annot_kws={'size': 8})
    ax.set_xlabel('Predicción', fontsize=10)
    ax.set_ylabel('Ground Truth', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)

plt.suptitle('Matrices de Confusión — DeepLabv3+ Test Set', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_confusion_antes_despues.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR / 'fig_confusion_antes_despues.png'}")

## 10. Figura 3 — Predicciones de muestra con postprocesamiento

In [ ]:
# Los vis_before y vis_after ya están calculados de §4
N_SHOW = 8
fig, axes = plt.subplots(N_SHOW, 4, figsize=(18, 4.5 * N_SHOW))
titles = ['Imagen', 'Ground Truth', 'DeepLabv3+ (sin PP)', 'DeepLabv3+ (con PP)']

for col, t in enumerate(titles):
    axes[0, col].set_title(t, fontsize=11, fontweight='bold')

for row in range(N_SHOW):
    img = vis_imgs[row].copy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    axes[row, 0].imshow(img)
    axes[row, 1].imshow(vis_gt[row],     cmap='tab10', vmin=0, vmax=8)
    axes[row, 2].imshow(vis_before[row], cmap='tab10', vmin=0, vmax=8)
    axes[row, 3].imshow(vis_after[row],  cmap='tab10', vmin=0, vmax=8)
    for col in range(4):
        axes[row, col].axis('off')

plt.suptitle('Predicciones con/sin Postprocesamiento — DeepLabv3+ Test Set',
             fontsize=13, fontweight='bold', y=1.001)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_sample_predictions_pp.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR / 'fig_sample_predictions_pp.png'}")

## 11. Resumen final

In [ ]:
print("=" * 70)
print("RESUMEN FINAL — POSTPROCESAMIENTO DeepLabv3+")
print("=" * 70)
print(f"  mIoU antes:    {sum_before['mIoU']:.4f}")
print(f"  mIoU después:  {sum_after['mIoU']:.4f}  ({sum_after['mIoU'] - sum_before['mIoU']:+.4f})")
print(f"  Pixel Acc antes:   {sum_before['pixel_accuracy']:.4f}")
print(f"  Pixel Acc después: {sum_after['pixel_accuracy']:.4f}")
print()
print(f"  {'Clase':<22} {'IoU Antes':>10} {'IoU Después':>12} {'Δ':>8}")
print("  " + "-" * 56)
for i in range(1, N_CLASSES):
    b = iou_before[i]; a = iou_after[i]
    tag = '✓' if (a - b) > 0.001 else ('=' if abs(a-b) <= 0.001 else '✗')
    print(f"  {CLASS_NAMES[i]:<22} {b:>10.4f} {a:>12.4f} {a-b:>+8.4f}  {tag}")
print()
archivos = [
    'test_metrics.json',
    'confusion_matrix_before.npy',
    'confusion_matrix_after.npy',
    'fig_iou_antes_despues.png',
    'fig_confusion_antes_despues.png',
    'fig_sample_predictions_pp.png',
    'visual_validation_subset.png',
]
print("Archivos generados:")
for f in archivos:
    p = OUTPUT_DIR / f
    s = '✓' if p.exists() else '✗'
    print(f"  [{s}] {p}")
print("=" * 70)